# Анализ производительности реализаций кеша
Сравнение `IMemoryCache`, `IDistributedCache` (Redis, Valkey, Garnet) и `HybridCache`.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re

CSV_PATH = "../RequestMonitoring.Benchmarks/bin/Release/net10.0/BenchmarkDotNet.Artifacts/results/RequestMonitoring.Benchmarks.CacheBenchmark-report.csv"

df = pd.read_csv(CSV_PATH, sep=";")
df = df[["Method", "Mean", "Error", "StdDev", "Allocated"]].copy()

def parse_ns(val):
    # Убираем кавычки, запятые, единицы измерения (ns, μs, ms и т.д.)
    val = str(val).replace('"', '').replace(',', '').strip()
    val = re.sub(r'[a-zA-Z\s]', '', val)
    return float(val)

for col in ["Mean", "Error", "StdDev"]:
    df[col] = df[col].apply(parse_ns)

df["Allocated_B"] = df["Allocated"].astype(str).str.replace(' B', '').str.replace(',', '').str.strip().astype(float)
df["Operation"] = df["Method"].apply(lambda x: "Get" if "Get" in x else "Set")
df["Cache"] = df["Method"].str.strip("'")
df["Mean_us"] = df["Mean"] / 1000
df["Error_us"] = df["Error"] / 1000

df.head(10)

In [ ]:
# График 1: Mean latency Get vs Set (логарифмическая шкала)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Get", "Set"))
colors = px.colors.qualitative.Plotly

for i, op in enumerate(["Get", "Set"]):
    subset = df[df["Operation"] == op].sort_values("Mean")
    fig.add_trace(
        go.Bar(
            x=subset["Cache"],
            y=subset["Mean_us"],
            error_y=dict(type="data", array=subset["Error_us"].tolist()),
            name=op,
            marker_color=colors[:len(subset)],
            showlegend=False
        ),
        row=1, col=i+1
    )

fig.update_yaxes(title_text="Среднее время (μs)", type="log")
fig.update_xaxes(tickangle=-30)
fig.update_layout(title_text="Latency реализаций кеша (логарифмическая шкала)", height=500)
fig.show()

In [ ]:
# График 2: Сравнение только distributed кешей (Redis, Valkey, Garnet)

distributed = df[df["Cache"].str.contains("Redis|Valkey|Garnet")].copy()

fig2 = px.bar(
    distributed.sort_values("Mean"),
    x="Cache",
    y="Mean_us",
    error_y="Error_us",
    color="Operation",
    barmode="group",
    title="Сравнение Redis / Valkey / Garnet",
    labels={"Mean_us": "Среднее время (μs)", "Cache": ""}
)
fig2.update_xaxes(tickangle=-30)
fig2.show()

In [ ]:
# График 3: Аллокации памяти

fig3 = px.bar(
    df.sort_values("Allocated_B"),
    x="Cache",
    y="Allocated_B",
    color="Operation",
    barmode="group",
    title="Аллокации памяти на операцию",
    labels={"Allocated_B": "Байт", "Cache": ""}
)
fig3.update_xaxes(tickangle=-30)
fig3.show()

In [ ]:
# Итоговая таблица

summary = df[["Cache", "Operation", "Mean_us", "Error_us", "Allocated_B"]].copy()
summary.columns = ["Реализация", "Операция", "Среднее (μs)", "Погрешность (μs)", "Аллокации (B)"]
summary = summary.sort_values("Среднее (μs)")
summary.style.background_gradient(subset=["Среднее (μs)"], cmap="RdYlGn_r")